In [ ]:
# Configuration
LOG_DIR = '/tmp/rs10-2mb'  # Change this to your log directory

In [57]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import re
from pathlib import Path
import glob
import os

In [58]:
def parse_logs(log_dir):
    """Parse logs from the specified directory and extract latency data."""
    
    latencies = []
    message_sizes = []
    nodes = []
    
    # Remove ANSI color codes regex
    ansi_escape = re.compile(r'\x1b\[[0-9;]*m')
    
    # Pattern to extract latency and message size
    pattern = re.compile(r'message received.*latency_ms=([0-9.]+).*message_size=([0-9]+)')
    
    # Read all stdout files
    for stdout_file in glob.glob(f'{log_dir}/*/stdout'):
        node_name = Path(stdout_file).parent.name
        
        with open(stdout_file, 'r') as f:
            for line in f:
                # Remove ANSI codes
                clean_line = ansi_escape.sub('', line)
                
                # Extract latency and message size
                match = pattern.search(clean_line)
                if match:
                    latency = float(match.group(1))
                    message_size = int(match.group(2)) / 1_000_000  # Convert to MB
                    
                    latencies.append(latency)
                    message_sizes.append(message_size)
                    nodes.append(node_name)
    
    # Create DataFrame
    df = pd.DataFrame({
        'latency_ms': latencies,
        'message_size_mb': message_sizes,
        'node': nodes
    })
    
    return df

# Load data
df = parse_logs(LOG_DIR)
print(f"Loaded {len(df)} data points from {LOG_DIR}")

Loaded 2176 data points from /tmp/rs16-2mb


In [59]:
# CDF plot for all nodes except g1
fig_cdf = go.Figure()

# Get unique nodes excluding g1
nodes_without_g1 = [node for node in df['node'].unique() if node != 'g1']
nodes_without_g1.sort()

print(f"Plotting {len(nodes_without_g1)} nodes (excluding g1)")

for node in nodes_without_g1:
    node_data = df[df['node'] == node]['latency_ms'].values
    sorted_latencies = np.sort(node_data)
    cdf_values = np.arange(1, len(sorted_latencies) + 1) / len(sorted_latencies)
    
    fig_cdf.add_trace(go.Scatter(
        x=sorted_latencies,
        y=cdf_values,
        mode='lines',
        name=node,
        line=dict(width=2)
    ))

fig_cdf.update_layout(
    title="CDF for All Nodes (excluding g1)",
    xaxis_title="Latency (ms)",
    yaxis_title="CDF",
    yaxis_tickformat=".1%",
    height=600,
    template="plotly_white"
)

fig_cdf.show()

Plotting 15 nodes (excluding g1)


In [60]:
# Histogram plot for all nodes except g1
fig_hist = go.Figure()

for node in nodes_without_g1:
    node_data = df[df['node'] == node]['latency_ms'].values
    
    fig_hist.add_trace(go.Histogram(
        x=node_data,
        name=node,
        opacity=0.7,
        nbinsx=30
    ))

fig_hist.update_layout(
    title="Histogram for All Nodes (excluding g1)",
    xaxis_title="Latency (ms)",
    yaxis_title="Count",
    barmode='overlay',
    height=600,
    template="plotly_white"
)

fig_hist.show()